# 11 — Full-information supervised price benchmark

This module pauses latent-regime development and asks a simpler question first:

**How much of the next five-minute Deribit index return can be predicted from the full causal market state we already collect and engineer?**

For a model row ending at time t, every predictor is measured no later than t-1:

\[
z_t = \alpha + \beta^\top X_{t-1} + \epsilon_t.
\]

The target is the same clock-scaled index return used by the regime notebooks. This is not a regression of the price level. Absolute price levels, identifiers, provenance fields and readiness flags are excluded as alpha variables.

Three estimators are used:

- OLS: transparent unregularized benchmark.
- Ridge: primary benchmark because the sensors are strongly collinear.
- Elastic Net: sparse challenger.

No missing value is mean-filled. A specification is trained and scored only where every predictor in that specification is observed. Pairwise model comparisons are recomputed on identical evaluation rows.

For comparability with Notebooks 08–10, training remains 2026-09-22 through 2026-09-28 and every later completed session is evaluation data. Because this evaluation period has already informed previous research decisions, Module 11 is exploratory rather than confirmatory.

Full information here means the economically meaningful engineered time_series_feature_state. Perpetual funding, raw contract-level option Greeks/OI and the new 1DTE chain are collected but are not yet propagated into this state table; they remain later extensions.

In [1]:
from pathlib import Path
import os, sys, time, warnings
import numpy as np
import pandas as pd
import duckdb
from dotenv import load_dotenv

warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 220)
pd.set_option("display.width", 260)

load_dotenv()
DB_PATH = Path(os.environ["DERIBIT_DATA_ROOT"]).expanduser().resolve() / "database" / "research.duckdb"
PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "src" / "data_engineering").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "src" / "data_engineering").exists():
    raise RuntimeError("Repository root not found from current working directory.")
SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from data_engineering.features import regime_inputs as ri
from data_engineering.modeling.clock_baseline import ClockBaselineSet, FeatureSpec
from data_engineering.modeling.hsmm.design import build_design
from data_engineering.modeling.hsmm.selection import blocked_folds
from data_engineering.modeling.hsmm.session_eval import paired_sessions
from data_engineering.modeling import full_information as fi

if fi.FULL_INFORMATION_VERSION < "0.1":
    raise RuntimeError("Old full_information.py: git pull and restart the kernel.")

ASSETS = ["AVAX", "BTC", "ETH", "HYPE", "SOL", "XRP"]
TRAIN_START, TRAIN_END = "2026-09-22", "2026-09-28"
MIN_FEATURE_COVERAGE = float(os.getenv("NB11_MIN_FEATURE_COVERAGE", "0.45"))
RIDGE_GRID = tuple(float(x) for x in os.getenv("NB11_RIDGE_GRID", "0.0001,0.001,0.01,0.1,1,10").split(","))
EN_ALPHA_GRID = tuple(float(x) for x in os.getenv("NB11_EN_ALPHA_GRID", "0.0001,0.001,0.01,0.1").split(","))
EN_L1_GRID = tuple(float(x) for x in os.getenv("NB11_EN_L1_GRID", "0.25,0.5,0.75").split(","))
N_BOOT = int(os.getenv("NB11_BOOT", "2000"))

print(
    f"full-information {fi.FULL_INFORMATION_VERSION} | train {TRAIN_START} -> {TRAIN_END} | "
    f"ridge {RIDGE_GRID} | EN {len(EN_ALPHA_GRID) * len(EN_L1_GRID)} candidates"
)

full-information 0.1 | train 2026-09-22 -> 2026-09-28 | ridge (0.0001, 0.001, 0.01, 0.1, 1.0, 10.0) | EN 12 candidates


## 1. Load the engineered state table and inventory the candidate information set

In [2]:
con = duckdb.connect(str(DB_PATH), read_only=True)
cols = set(con.execute("DESCRIBE time_series_feature_state").df()["column_name"])

required = {
    "state_id", "state_time", "underlying", "session_id", "return_5m", "futures_index_price",
    "is_weekend", "utc_minute_of_day", "session_progress_fraction", "settlement_locked_fraction",
    "in_settlement_window", "vwap", "volume_usd", "futures_tradable_mid_price", "trade_count",
    "perp_integrity_ok", "known_exchange_event", "integrity_flag_trades_outside_bbo",
    "microstructure_features_ready", "instability_features_ready", "option_snapshot_ready",
    "option_atm_quote_ok", "option_shape_quote_ok", "fit_ok",
}
candidate_cols = {s.column for s in fi.FEATURE_SPECS}
want = sorted((required | candidate_cols) & cols)

df = con.execute(
    f"SELECT {', '.join(want)} FROM time_series_feature_state ORDER BY underlying, state_time"
).df()
df["state_time"] = pd.to_datetime(df["state_time"], utc=True)
df = df[df["underlying"].isin(ASSETS)].reset_index(drop=True)
df["r"] = pd.to_numeric(df["return_5m"], errors="coerce")
df["session_date"] = ri.session_key(df["state_time"])

dates = sorted(df.loc[df["r"].notna(), "session_date"].unique())
TRAIN_DATES = {d for d in dates if TRAIN_START <= str(d) <= TRAIN_END}
TEST_DATES = {d for d in dates if str(d) > TRAIN_END}
if len(TRAIN_DATES) < 5 or not TEST_DATES:
    raise RuntimeError("Module 11 needs the Sep 22-28 discovery sessions and at least one later session.")

print(
    f"Rows {len(df):,} | sessions {len(dates)} | "
    f"train {min(TRAIN_DATES)} -> {max(TRAIN_DATES)} ({len(TRAIN_DATES)}) | "
    f"evaluation {min(TEST_DATES)} -> {max(TEST_DATES)} ({len(TEST_DATES)})"
)
print(
    "Candidate economic features in schema:",
    sum(s.column in cols for s in fi.FEATURE_SPECS),
    "/",
    len(fi.FEATURE_SPECS),
)

missing_schema = [
    (s.block, s.name, s.column)
    for s in fi.FEATURE_SPECS
    if s.column not in cols
]
if missing_schema:
    print("Not yet present in time_series_feature_state:")
    display(pd.DataFrame(missing_schema, columns=["block", "feature", "column"]))

Rows 31,104 | sessions 16 | train 2026-09-22 -> 2026-09-28 (7) | evaluation 2026-09-29 -> 2026-10-07 (9)
Candidate economic features in schema: 73 / 73


## 2. Build causal model matrices

The session-VWAP displacement and clock-scaled target are rebuilt exactly as in Notebooks 08–10. Raw state features are aligned one bar backward using the design source row.

Feature eligibility is decided only from the discovery training sessions. A feature needs at least 45% usable training coverage for that asset. This keeps sparse sensors from collapsing the full complete-case sample while still allowing quote-backed option information to participate.

In [3]:
def build_asset_bundle(asset):
    g = df[df.underlying == asset].copy().reset_index(drop=True)
    train_state = g["session_date"].isin(TRAIN_DATES)

    clock_frame = g[["state_id", "state_time", "underlying", "is_weekend", "return_5m"]]
    cb = ClockBaselineSet.fit(
        clock_frame,
        train_state,
        specs=(FeatureSpec("return_5m", "return_5m", transform="log_abs", scale="none"),),
    )
    f_clock = (
        cb.transform(clock_frame)["return_5m__clock_vol_factor"]
        .fillna(1.0)
        .to_numpy()
    )
    d = build_design(g, f_clock)

    raw, used_specs, missing = fi.raw_feature_frame(g)
    clock = fi.clock_core(g)
    src = d.frame["source_row"].to_numpy(int)
    pred_row = src - 1

    X = pd.DataFrame(index=np.arange(len(d.frame)))
    for c in ["D", "z1", "z2", "z3", "vr_q6_prev"]:
        X[c] = pd.to_numeric(d.frame[c], errors="coerce").to_numpy()
    for c in clock.columns:
        X[c] = clock.iloc[pred_row][c].to_numpy()
    for c in raw.columns:
        X[c] = raw.iloc[pred_row][c].to_numpy()

    y = d.frame["y"].to_numpy(float)
    sessions = d.frame["session_date"].to_numpy()
    tr = np.isin(sessions, list(TRAIN_DATES))
    te = np.isin(sessions, list(TEST_DATES))

    coverage = X.loc[tr].notna().mean()
    eligible_specs = [
        s
        for s in used_specs
        if coverage.get(s.name, 0.0) >= MIN_FEATURE_COVERAGE
        and X.loc[tr, s.name].nunique(dropna=True) >= 3
    ]
    blocks = fi.block_map(eligible_specs)
    core = [c for c in fi.CORE_NAMES if c in X]

    market_blocks = ["returns", "volatility", "flow", "basis_oi"]
    nested = {
        "M0 Core": core,
        "M1 +Market": core + sum((blocks.get(b, []) for b in market_blocks), []),
        "M2 +L2": core + sum((blocks.get(b, []) for b in market_blocks + ["l2"]), []),
        "M3 Full": core + sum((blocks.get(b, []) for b in market_blocks + ["l2", "options"]), []),
    }
    nested = {k: list(dict.fromkeys(v)) for k, v in nested.items()}

    return dict(
        asset=asset,
        g=g,
        design=d,
        X=X,
        y=y,
        sessions=sessions,
        train=tr,
        test=te,
        specs=eligible_specs,
        blocks=blocks,
        nested=nested,
        coverage=coverage,
    )

BUNDLES = {a: build_asset_bundle(a) for a in ASSETS}

inventory = []
for a, b in BUNDLES.items():
    for s in b["specs"]:
        inventory.append(
            {
                "underlying": a,
                "block": s.block,
                "feature": s.name,
                "train_coverage": b["coverage"][s.name],
            }
        )

inv = pd.DataFrame(inventory)
print("Eligible engineered predictors by block:")
display(
    inv.groupby(["underlying", "block"])
    .agg(
        n=("feature", "size"),
        min_cov=("train_coverage", "min"),
        med_cov=("train_coverage", "median"),
    )
    .round(3)
)

print("Nested model sizes:")
display(
    pd.DataFrame(
        {
            a: {k: len(v) for k, v in b["nested"].items()}
            for a, b in BUNDLES.items()
        }
    ).T
)

print("Complete-case training/evaluation coverage:")
coverage_rows = []
for a, b in BUNDLES.items():
    for name, names in b["nested"].items():
        sc = fi.fit_standardizer(b["X"], names, b["train"])
        Z = sc.transform(b["X"])
        coverage_rows.append(
            {
                "underlying": a,
                "model": name,
                "n_features": len(names),
                "train_rows": int(np.sum(b["train"] & fi.complete_rows(Z, b["y"]))),
                "test_rows": int(np.sum(b["test"] & fi.complete_rows(Z, b["y"]))),
            }
        )
display(pd.DataFrame(coverage_rows))

Eligible engineered predictors by block:


n  min_cov  med_cov
underlying block                           
AVAX       basis_oi     6    0.862    1.000
           flow         6    0.958    0.979
           l2          11    0.862    0.862
           options     33    0.673    0.805
           returns      4    1.000    1.000
           volatility   4    0.763    0.979
BTC        basis_oi     6    0.862    1.000
           flow         6    1.000    1.000
           l2          11    0.862    0.862
           options     31    0.452    0.573
           returns      4    1.000    1.000
           volatility   4    0.764    1.000
ETH        basis_oi     6    0.862    1.000
           flow         6    1.000    1.000
           l2          11    0.862    0.862
           options     33    0.501    0.709
           returns      4    1.000    1.000
           volatility   4    0.764    1.000
HYPE       basis_oi     6    0.716    1.000
           flow         6    0.716    0.858
           l2          11    0.862    0.862
           options     33    0.577    0.729
           returns      4    1.000    1.000
           volatility   4    0.716    0.882
SOL        basis_oi     6    0.862    1.000
           flow         6    1.000    1.000
           l2          11    0.862    0.862
           options     33    0.476    0.735
           returns      4    1.000    1.000
           volatility   4    0.763    1.000
XRP        basis_oi     6    0.862    1.000
           flow         6    0.940    0.970
           l2          11    0.862    0.862
           options     33    0.521    0.657
           returns      4    1.000    1.000
           volatility   4    0.763    0.970

Nested model sizes:


,M0 Core,M1 +Market,M2 +L2,M3 Full
AVAX,12,32,43,76
BTC,12,32,43,74
ETH,12,32,43,76
HYPE,12,32,43,76
SOL,12,32,43,76
XRP,12,32,43,76


Complete-case training/evaluation coverage:


,underlying,model,n_features,train_rows,test_rows
0,AVAX,M0 Core,12,1252,1337
1,AVAX,M1 +Market,32,1008,1198
2,AVAX,M2 +L2,43,1008,1198
3,AVAX,M3 Full,76,561,624
4,BTC,M0 Core,12,1253,1340
5,BTC,M1 +Market,32,1059,1340
6,BTC,M2 +L2,43,1059,1340
7,BTC,M3 Full,74,313,254
8,ETH,M0 Core,12,1253,1342
9,ETH,M1 +Market,32,1059,1342


# Part A — Training-only regularization selection

Ridge and Elastic Net hyperparameters are selected with blocked expanding-window cross-validation inside Sep 22–28 only. The objective is mean Gaussian log score per validation session.

Preprocessing is refitted inside every fold. Missing rows are dropped, never imputed.

In [4]:
def cv_candidates(bundle, names, candidates):
    sessions = bundle["sessions"]
    folds = blocked_folds(sorted(TRAIN_DATES), n_folds=3, min_train_share=0.45)
    rows = []

    for ci, cand in enumerate(candidates):
        sess_ll = {}
        valid_fits = 0

        for fold_i, (tr_dates, va_dates) in enumerate(folds):
            itr = np.isin(sessions, list(tr_dates))
            iva = np.isin(sessions, list(va_dates))

            scaler = fi.fit_standardizer(bundle["X"], names, itr)
            Z = scaler.transform(bundle["X"])

            try:
                model = fi.fit_linear(
                    Z[itr],
                    bundle["y"][itr],
                    names,
                    **cand,
                )
            except ValueError:
                continue

            ok = iva & fi.complete_rows(Z, bundle["y"])
            if not ok.any():
                continue

            valid_fits += 1
            pred = model.predict(Z[ok])
            ll = fi.gaussian_logscore(bundle["y"][ok], pred, model.sigma)

            for sd, value in pd.Series(ll).groupby(sessions[ok]).sum().items():
                sess_ll[sd] = sess_ll.get(sd, 0.0) + float(value)

        score = (
            float(np.mean(list(sess_ll.values())))
            if sess_ll and valid_fits
            else -np.inf
        )
        rows.append(
            {
                "candidate": ci,
                "cv_mean_session_ll": score,
                "valid_folds": valid_fits,
                **cand,
            }
        )

    tab = pd.DataFrame(rows).sort_values("cv_mean_session_ll", ascending=False)
    if tab.empty or not np.isfinite(tab.iloc[0]["cv_mean_session_ll"]):
        raise RuntimeError(f"No valid CV fit for {bundle['asset']} with {len(names)} predictors")
    return tab.iloc[0].to_dict(), tab

RIDGE_CANDS = [
    dict(kind="ridge", alpha=a, l1_ratio=0.0)
    for a in RIDGE_GRID
]
EN_CANDS = [
    dict(kind="elasticnet", alpha=a, l1_ratio=l)
    for a in EN_ALPHA_GRID
    for l in EN_L1_GRID
]

TUNING = {}
for a, b in BUNDLES.items():
    TUNING[a] = {}
    t0 = time.perf_counter()

    for model_name, names in b["nested"].items():
        best, tab = cv_candidates(b, names, RIDGE_CANDS)
        TUNING[a][("ridge", model_name)] = (best, tab)

    best, tab = cv_candidates(b, b["nested"]["M3 Full"], EN_CANDS)
    TUNING[a][("elasticnet", "M3 Full")] = (best, tab)

    print(f"{a}: tuning complete in {time.perf_counter() - t0:.1f}s")

selected = []
for a, tuning in TUNING.items():
    for (kind, model_name), (best, tab) in tuning.items():
        selected.append(
            {
                "underlying": a,
                "kind": kind,
                "model": model_name,
                "alpha": best["alpha"],
                "l1_ratio": best["l1_ratio"],
                "cv_mean_session_ll": best["cv_mean_session_ll"],
            }
        )

selected = pd.DataFrame(selected)
display(selected.round(4))

AVAX: tuning complete in 166.6s
BTC: tuning complete in 166.1s
ETH: tuning complete in 113.7s
HYPE: tuning complete in 109.4s
SOL: tuning complete in 107.6s
XRP: tuning complete in 114.1s


,underlying,kind,model,alpha,l1_ratio,cv_mean_session_ll
0,AVAX,ridge,M0 Core,10.0,0.00,-309.1196
1,AVAX,ridge,M1 +Market,10.0,0.00,-304.3205
2,AVAX,ridge,M2 +L2,10.0,0.00,-304.5994
3,AVAX,ridge,M3 Full,10.0,0.00,-153.8189
4,AVAX,elasticnet,M3 Full,0.1,0.75,-154.2654
5,BTC,ridge,M0 Core,10.0,0.00,-296.7829
6,BTC,ridge,M1 +Market,1.0,0.00,-296.4704
7,BTC,ridge,M2 +L2,1.0,0.00,-296.4674
8,BTC,ridge,M3 Full,10.0,0.00,-113.0557
9,BTC,elasticnet,M3 Full,0.1,0.75,-112.1987


# Part B — Final chronological evaluation

Each final model is fitted on all Sep 22–28 discovery rows and scored on every later session.

The primary nested Ridge comparison is:

\[
M0\ Core \rightarrow M1\ Market \rightarrow M2\ L2 \rightarrow M3\ Full.
\]

For every pair, scores are compared only on the intersection of rows on which both models can make a prediction.

In [5]:
def fit_final(bundle, names, kind, alpha=0.0, l1_ratio=0.0):
    scaler = fi.fit_standardizer(bundle["X"], names, bundle["train"])
    Z = scaler.transform(bundle["X"])

    model = fi.fit_linear(
        Z[bundle["train"]],
        bundle["y"][bundle["train"]],
        names,
        kind=kind,
        alpha=alpha,
        l1_ratio=l1_ratio,
    )

    ok = bundle["test"] & fi.complete_rows(Z, bundle["y"])
    pred = np.full(len(bundle["y"]), np.nan)
    score = np.full(len(bundle["y"]), np.nan)

    pred[ok] = model.predict(Z[ok])
    score[ok] = fi.gaussian_logscore(
        bundle["y"][ok],
        pred[ok],
        model.sigma,
    )

    metrics = fi.oos_metrics(
        bundle["y"][ok],
        pred[ok],
        np.nanmean(bundle["y"][bundle["train"]]),
    )

    return dict(
        model=model,
        scaler=scaler,
        pred=pred,
        score=score,
        ok=ok,
        metrics=metrics,
    )

RESULTS = {}
for a, b in BUNDLES.items():
    out = {}

    for model_name, names in b["nested"].items():
        best = TUNING[a][("ridge", model_name)][0]
        out[("ridge", model_name)] = fit_final(
            b,
            names,
            "ridge",
            best["alpha"],
            0.0,
        )

    full = b["nested"]["M3 Full"]
    out[("ols", "M3 Full")] = fit_final(b, full, "ols")

    best = TUNING[a][("elasticnet", "M3 Full")][0]
    out[("elasticnet", "M3 Full")] = fit_final(
        b,
        full,
        "elasticnet",
        best["alpha"],
        best["l1_ratio"],
    )

    RESULTS[a] = out

def paired_result(bundle, A, B):
    common = A["ok"] & B["ok"]
    return paired_sessions(
        A["score"][common],
        B["score"][common],
        bundle["sessions"][common],
        n_boot=N_BOOT,
    )

nested_rows = []
for a, b in BUNDLES.items():
    r = RESULTS[a]

    for A, B in [
        ("M1 +Market", "M0 Core"),
        ("M2 +L2", "M1 +Market"),
        ("M3 Full", "M2 +L2"),
        ("M3 Full", "M0 Core"),
    ]:
        nested_rows.append(
            {
                "underlying": a,
                "model": A,
                "vs": B,
                **paired_result(
                    b,
                    r[("ridge", A)],
                    r[("ridge", B)],
                ),
            }
        )

nested_cmp = pd.DataFrame(nested_rows)
SHOW = [
    "underlying", "model", "vs", "n_sessions",
    "mean_session_diff", "boot_lo", "boot_hi",
    "median_session_diff", "sessions_won",
    "loso_min", "loso_max",
    "per_bar_x1000", "per_bar_nw_t",
]
display(nested_cmp[SHOW].round(3))

metric_rows = []
for a, b in BUNDLES.items():
    for key, result in RESULTS[a].items():
        kind, model_name = key
        metric_rows.append(
            {
                "underlying": a,
                "kind": kind,
                "model": model_name,
                "n_features": len(result["model"].feature_names),
                "n_train": result["model"].n_train,
                "rank": result["model"].rank,
                "condition": result["model"].condition_number,
                **result["metrics"],
            }
        )

metrics = pd.DataFrame(metric_rows)
print("OOS predictive metrics:")
display(metrics.round(4))

,underlying,model,vs,n_sessions,mean_session_diff,boot_lo,boot_hi,median_session_diff,sessions_won,loso_min,loso_max,per_bar_x1000,per_bar_nw_t
0,AVAX,M1 +Market,M0 Core,9,0.178,-0.101,0.441,0.131,0.667,0.106,0.270,1.339,1.277
1,AVAX,M2 +L2,M1 +Market,9,-0.068,-0.162,0.026,-0.083,0.222,-0.095,-0.041,-0.512,-1.688
2,AVAX,M3 Full,M2 +L2,9,-0.968,-2.310,0.280,0.004,0.556,-1.362,-0.508,-13.956,-1.496
3,AVAX,M3 Full,M0 Core,9,-0.810,-2.254,0.486,0.207,0.556,-1.221,-0.323,-11.685,-1.180
4,BTC,M1 +Market,M0 Core,9,-0.723,-1.608,0.119,-0.487,0.444,-0.910,-0.402,-4.856,-1.595
5,BTC,M2 +L2,M1 +Market,9,-0.018,-0.175,0.126,0.020,0.556,-0.065,0.048,-0.119,-0.173
6,BTC,M3 Full,M2 +L2,7,-0.152,-0.486,0.133,0.141,0.571,-0.220,-0.027,-4.198,-0.503
7,BTC,M3 Full,M0 Core,7,-0.035,-0.510,0.355,0.290,0.571,-0.151,0.153,-0.961,-0.142
8,ETH,M1 +Market,M0 Core,9,-0.067,-0.238,0.071,0.019,0.556,-0.103,0.007,-0.447,-0.881
9,ETH,M2 +L2,M1 +Market,9,-0.059,-0.125,0.014,-0.076,0.333,-0.086,-0.036,-0.395,-1.169


OOS predictive metrics:


,underlying,kind,model,n_features,n_train,rank,condition,n,mse,rmse,r2,corr,sign_accuracy
0,AVAX,ridge,M0 Core,12,1252,12,5.356100e+00,1337,1.1477,1.0713,-0.0001,0.0543,0.4936
1,AVAX,ridge,M1 +Market,32,1008,31,6.006536e+15,1198,1.1690,1.0812,0.0027,0.0453,0.5067
2,AVAX,ridge,M2 +L2,43,1008,42,6.140213e+15,1198,1.1701,1.0817,0.0017,0.0385,0.5192
3,AVAX,ridge,M3 Full,76,561,72,2.180842e+16,624,1.1901,1.0909,0.0038,0.0406,0.5096
4,AVAX,ols,M3 Full,76,561,72,2.180842e+16,624,1.6396,1.2805,-0.3724,-0.0072,0.4792
5,AVAX,elasticnet,M3 Full,76,561,72,2.180842e+16,624,1.1913,1.0915,0.0028,NaN,0.5016
6,BTC,ridge,M0 Core,12,1253,12,5.084300e+00,1340,1.2759,1.1296,-0.0003,0.0311,0.5075
7,BTC,ridge,M1 +Market,32,1059,31,1.103294e+16,1340,1.2846,1.1334,-0.0071,0.0050,0.5082
8,BTC,ridge,M2 +L2,43,1059,42,1.027071e+16,1340,1.2844,1.1333,-0.0069,0.0123,0.5045
9,BTC,ridge,M3 Full,74,313,69,1.324431e+31,254,1.0778,1.0382,-0.0004,-0.0029,0.5039


## Part C — OLS vs Ridge vs Elastic Net on identical full-information rows

In [6]:
est_rows = []
for a, b in BUNDLES.items():
    r = RESULTS[a]
    ridge = r[("ridge", "M3 Full")]

    for kind in ["ols", "elasticnet"]:
        challenger = r[(kind, "M3 Full")]
        est_rows.append(
            {
                "underlying": a,
                "model": kind,
                "vs": "ridge",
                **paired_result(b, challenger, ridge),
            }
        )

est_cmp = pd.DataFrame(est_rows)
display(est_cmp[SHOW].round(3))

print("Elastic Net sparsity:")
sparsity = []
for a in ASSETS:
    m = RESULTS[a][("elasticnet", "M3 Full")]["model"]
    sparsity.append(
        {
            "underlying": a,
            "features": len(m.coef),
            "nonzero": int(np.sum(np.abs(m.coef) > 1e-10)),
            "alpha": m.alpha,
            "l1_ratio": m.l1_ratio,
        }
    )
display(pd.DataFrame(sparsity))

,underlying,model,vs,n_sessions,mean_session_diff,boot_lo,boot_hi,median_session_diff,sessions_won,loso_min,loso_max,per_bar_x1000,per_bar_nw_t
0,AVAX,ols,ridge,9,-19.469,-34.755,-7.308,-9.818,0.000,-21.867,-13.521,-280.808,-3.702
1,AVAX,elasticnet,ridge,9,0.025,-0.165,0.224,-0.073,0.444,-0.037,0.074,0.362,0.262
2,BTC,ols,ridge,7,-73.877,-124.111,-27.944,-23.033,0.000,-84.537,-56.574,-2035.987,-4.727
3,BTC,elasticnet,ridge,7,-0.159,-0.778,0.299,-0.068,0.429,-0.309,0.120,-4.375,-0.713
4,ETH,ols,ridge,7,-274.457,-734.724,-27.763,-39.098,0.000,-318.618,-49.304,-5601.168,-1.294
5,ETH,elasticnet,ridge,7,-0.129,-0.538,0.206,-0.077,0.429,-0.224,0.062,-2.634,-0.674
6,HYPE,ols,ridge,8,-93.938,-213.189,-20.351,-25.983,0.000,-106.356,-37.303,-1747.677,-5.529
7,HYPE,elasticnet,ridge,8,0.210,-0.047,0.515,0.046,0.500,0.087,0.287,3.910,1.089
8,SOL,ols,ridge,6,-67.039,-110.663,-27.600,-53.719,0.000,-78.239,-49.494,-2628.996,-4.895
9,SOL,elasticnet,ridge,6,0.012,-0.076,0.127,-0.028,0.333,-0.043,0.041,0.453,0.078


Elastic Net sparsity:


,underlying,features,nonzero,alpha,l1_ratio
0,AVAX,76,0,0.1,0.75
1,BTC,74,8,0.1,0.75
2,ETH,76,7,0.1,0.75
3,HYPE,76,12,0.1,0.75
4,SOL,76,8,0.1,0.75
5,XRP,76,4,0.1,0.75


# Part D — Family ablations

For each economic block, the full Ridge model is refitted without that block and compared with the full model on identical held-out rows.

A positive Full-minus-without-block score means that block adds unique out-of-sample price information after the other measurements are already present.

In [7]:
ABLATIONS = {}
ab_rows = []

for a, b in BUNDLES.items():
    full_names = b["nested"]["M3 Full"]
    full = RESULTS[a][("ridge", "M3 Full")]
    ABLATIONS[a] = {}

    for block, block_names in b["blocks"].items():
        reduced = [
            x
            for x in full_names
            if x not in set(block_names)
        ]

        if len(reduced) == len(full_names):
            continue

        best, tab = cv_candidates(
            b,
            reduced,
            RIDGE_CANDS,
        )

        reduced_result = fit_final(
            b,
            reduced,
            "ridge",
            best["alpha"],
            0.0,
        )

        ABLATIONS[a][block] = reduced_result
        ab_rows.append(
            {
                "underlying": a,
                "block": block,
                "n_removed": len(full_names) - len(reduced),
                **paired_result(
                    b,
                    full,
                    reduced_result,
                ),
            }
        )

ablation = pd.DataFrame(ab_rows)
display(
    ablation[
        [
            "underlying", "block", "n_removed", "n_sessions",
            "mean_session_diff", "boot_lo", "boot_hi",
            "sessions_won", "loso_min", "loso_max",
            "per_bar_x1000", "per_bar_nw_t",
        ]
    ].round(3)
)

,underlying,block,n_removed,n_sessions,mean_session_diff,boot_lo,boot_hi,sessions_won,loso_min,loso_max,per_bar_x1000,per_bar_nw_t
0,AVAX,returns,4,9,0.011,-0.029,0.049,0.667,-0.001,0.021,0.153,0.434
1,AVAX,volatility,4,9,0.079,0.018,0.149,0.778,0.057,0.095,1.146,2.326
2,AVAX,flow,6,9,-0.017,-0.060,0.023,0.556,-0.031,-0.005,-0.250,-0.921
3,AVAX,basis_oi,6,9,-0.002,-0.026,0.019,0.556,-0.008,0.006,-0.035,-0.135
4,AVAX,l2,11,9,-0.003,-0.076,0.080,0.556,-0.036,0.021,-0.043,-0.085
5,AVAX,options,33,9,-0.968,-2.310,0.280,0.556,-1.362,-0.508,-13.956,-1.496
6,BTC,returns,4,7,-0.017,-0.161,0.106,0.429,-0.054,0.034,-0.465,-0.252
7,BTC,volatility,4,7,-0.008,-0.031,0.008,0.429,-0.012,0.002,-0.224,-1.274
8,BTC,flow,6,7,-0.002,-0.069,0.054,0.714,-0.016,0.026,-0.051,-0.071
9,BTC,basis_oi,6,7,-0.001,-0.083,0.066,0.714,-0.020,0.032,-0.037,-0.040


# Part E — Which measurements does the regularized model use?

Every predictor is standardized using training-only moments, so coefficient magnitudes are comparable within an asset. These coefficients are descriptive predictive weights, not causal effects.

In [8]:
coef_rows = []

for a, b in BUNDLES.items():
    for kind in ["ridge", "elasticnet", "ols"]:
        m = RESULTS[a][(kind, "M3 Full")]["model"]

        for name, value in zip(m.feature_names, m.coef):
            block = "core"

            for block_name, block_features in b["blocks"].items():
                if name in block_features:
                    block = block_name
                    break

            coef_rows.append(
                {
                    "underlying": a,
                    "kind": kind,
                    "feature": name,
                    "block": block,
                    "coef": value,
                    "abs_coef": abs(value),
                }
            )

coefs = pd.DataFrame(coef_rows)

for a in ASSETS:
    print(f"\n{a} - top standardized Ridge coefficients")
    display(
        coefs[
            (coefs.underlying == a)
            & (coefs.kind == "ridge")
        ]
        .nlargest(15, "abs_coef")
        [["feature", "block", "coef"]]
        .round(4)
    )

print("Median absolute Ridge coefficient by block:")
display(
    coefs[coefs.kind == "ridge"]
    .groupby(["underlying", "block"])
    .abs_coef.median()
    .unstack()
    .round(4)
)


AVAX - top standardized Ridge coefficients


,feature,block,coef
59,skew025_d15,options,-0.0046
15,perp_ret_5m,returns,-0.0043
1,z1,core,-0.0043
18,log_rv_4h,volatility,-0.0042
41,obi_20_min,l2,0.0040
13,ret_30m,returns,-0.0038
56,atm_iv_d30,options,-0.0038
29,microprice_mid_bps,basis_oi,-0.0037
38,log_bid_depth20,l2,0.0036
25,log_avg_trade,flow,0.0036



BTC - top standardized Ridge coefficients


,feature,block,coef
241,ret_30m,returns,-0.0119
242,ret_1h,returns,-0.0116
229,z1,core,-0.0107
243,perp_ret_5m,returns,-0.0103
230,z2,core,0.0098
255,perp_mid_basis_bps,basis_oi,0.0097
228,D,core,-0.0075
250,tfi,flow,0.0067
251,buy_share,flow,0.0067
287,skew025_d15,options,-0.0058



ETH - top standardized Ridge coefficients


,feature,block,coef
452,z2,core,0.0131
463,ret_30m,returns,-0.0111
453,z3,core,-0.0101
464,ret_1h,returns,-0.0101
523,log_option_spread_med,options,0.0086
488,log_bid_depth20,l2,-0.0076
508,skew025_d5,options,0.0064
485,obi_10,l2,-0.0064
506,atm_iv_d30,options,-0.0058
486,obi_20,l2,-0.0055


... (4 outputs left out)


XRP - top standardized Ridge coefficients


,feature,block,coef
1200,rr25_d5,options,-0.0080
1148,ret_1h,returns,-0.0073
1136,z2,core,0.0072
1195,skew025_d60,options,-0.0068
1172,log_bid_depth20,l2,-0.0065
1192,skew025_d5,options,-0.0060
1160,close_vs_vwap_bps,basis_oi,0.0057
1147,ret_30m,returns,-0.0053
1197,curv025_d15,options,0.0051
1163,microprice_mid_bps,basis_oi,0.0050


Median absolute Ridge coefficient by block:


block,basis_oi,core,flow,l2,options,returns,volatility
underlying,,,,,,,
AVAX,0.0018,0.0007,0.0020,0.0017,0.0018,0.0032,0.0017
BTC,0.0021,0.0025,0.0032,0.0026,0.0015,0.0109,0.0010
ETH,0.0011,0.0021,0.0032,0.0023,0.0024,0.0067,0.0029
HYPE,0.0031,0.0026,0.0026,0.0022,0.0033,0.0036,0.0033
SOL,0.0037,0.0031,0.0035,0.0069,0.0022,0.0042,0.0039
XRP,0.0027,0.0026,0.0015,0.0024,0.0022,0.0045,0.0017


# Part F — Interpretation gates

The benchmark earns a role only through chronological out-of-sample performance.

1. Full Ridge vs Core asks whether the complete information set improves next-return prediction at all.
2. Family ablations identify which sensor blocks add unique information.
3. If Ridge materially beats OLS, multicollinearity and estimation variance matter and raw OLS coefficients should not be interpreted structurally.
4. A sparse Elastic-Net win would identify a smaller supervised feature contract.
5. If the full model fails, the collected state may still be useful for volatility, option valuation, observation quality or transition risk; it simply does not justify a directional five-minute alpha claim.

This module does not search regimes. A switching model should return only if the supervised mapping from market state to future return itself shows instability or if a switching specification beats this benchmark chronologically.

In [9]:
summary = []

for a, b in BUNDLES.items():
    full_vs_core = nested_cmp[
        (nested_cmp.underlying == a)
        & (nested_cmp.model == "M3 Full")
        & (nested_cmp.vs == "M0 Core")
    ].iloc[0]

    ridge = metrics[
        (metrics.underlying == a)
        & (metrics.kind == "ridge")
        & (metrics.model == "M3 Full")
    ].iloc[0]

    ols = metrics[
        (metrics.underlying == a)
        & (metrics.kind == "ols")
        & (metrics.model == "M3 Full")
    ].iloc[0]

    en = metrics[
        (metrics.underlying == a)
        & (metrics.kind == "elasticnet")
        & (metrics.model == "M3 Full")
    ].iloc[0]

    summary.append(
        {
            "underlying": a,
            "full_vs_core_mean_session_ll": full_vs_core.mean_session_diff,
            "full_vs_core_boot_lo": full_vs_core.boot_lo,
            "full_vs_core_boot_hi": full_vs_core.boot_hi,
            "sessions_won": full_vs_core.sessions_won,
            "ridge_oos_r2": ridge.r2,
            "ridge_oos_corr": ridge["corr"],
            "ridge_sign_accuracy": ridge.sign_accuracy,
            "ols_oos_r2": ols.r2,
            "elasticnet_oos_r2": en.r2,
            "full_features": int(ridge.n_features),
            "full_test_rows": int(ridge.n),
        }
    )

module11_summary = pd.DataFrame(summary)
display(module11_summary.round(4))

nb11_report = {
    "feature_inventory": inv,
    "tuning": selected,
    "nested_comparison": nested_cmp,
    "metrics": metrics,
    "estimator_comparison": est_cmp,
    "ablation": ablation,
    "coefficients": coefs,
    "summary": module11_summary,
}

for k, v in nb11_report.items():
    print(f"{k:<24} {len(v):>6,} rows")

con.close()
print("DuckDB connection closed.")

,underlying,full_vs_core_mean_session_ll,full_vs_core_boot_lo,full_vs_core_boot_hi,sessions_won,ridge_oos_r2,ridge_oos_corr,ridge_sign_accuracy,ols_oos_r2,elasticnet_oos_r2,full_features,full_test_rows
0,AVAX,-0.8101,-2.2541,0.4860,0.5556,0.0038,0.0406,0.5096,-0.3724,0.0028,76,624
1,BTC,-0.0349,-0.5100,0.3552,0.5714,-0.0004,-0.0029,0.5039,-2.8934,-0.0066,74,254
2,ETH,0.1399,-0.8683,0.8341,0.8571,0.0040,0.0564,0.5714,-8.2050,-0.0003,76,343
3,HYPE,-0.0572,-0.7019,0.4852,0.7500,-0.0023,0.0339,0.5047,-2.3885,0.0056,76,430
4,SOL,0.0265,-0.4268,0.5531,0.5000,0.0058,0.1043,0.4837,-2.4166,0.0066,76,153
5,XRP,-1.2552,-2.3881,-0.3278,0.1429,-0.0040,-0.0492,0.5256,-0.4250,-0.0014,76,215


feature_inventory           382 rows
tuning                       30 rows
nested_comparison            24 rows
metrics                      36 rows
estimator_comparison         12 rows
ablation                     36 rows
coefficients              1,362 rows
summary                       6 rows
DuckDB connection closed.
